# 006 - EDGAR Pipeline Diagnostic

Step-by-step diagnostic of the SEC EDGAR quarterly fundamentals pipeline.
Tests data extraction, cumulative differencing, and final output for multiple tickers.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.data_fetch.fetch_fundamentals_quarterly import (
    _load_cik_mapping, _fetch_company_facts, _extract_fact_series,
    _extract_quarterly_duration, _extract_quarterly_instant,
    _build_quarterly_dataframe, XBRL_TAG_MAP
)

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

cik_map = _load_cik_mapping()
print(f'CIK mapping loaded: {len(cik_map)} tickers')

## Step 1: Fetch raw EDGAR data for test tickers

In [ ]:
TEST_TICKERS = ['AAPL', 'MSFT', 'JPM', 'AMZN', 'WMT']

facts_cache = {}
for ticker in TEST_TICKERS:
    cik = cik_map.get(ticker)
    if cik:
        facts = _fetch_company_facts(cik)
        facts_cache[ticker] = facts
        namespaces = list(facts.get('facts', {}).keys()) if facts else []
        print(f'{ticker} (CIK {cik}): namespaces={namespaces}')
    else:
        print(f'{ticker}: CIK not found')

## Step 2: Raw fact extraction - Revenue tags

Show which XBRL tags are available for revenue, and the raw entries from EDGAR.

In [ ]:
ticker = 'AAPL'
facts = facts_cache[ticker]

print(f'Revenue tags to try: {XBRL_TAG_MAP["revenue"]}')
print()

tags = XBRL_TAG_MAP['revenue']
raw = _extract_fact_series(facts, tags, unit='USD',
                           form_filter=['10-Q', '10-K', '10-Q/A', '10-K/A'])
print(f'{ticker} raw revenue entries: {len(raw)}')
print(f'Date range: {raw["end"].min()} to {raw["end"].max()}')

# Show entries for the most recent fiscal year
recent = raw[raw['end'] >= '2024-09-01'].sort_values('end')
if 'start' in recent.columns:
    recent['duration_days'] = (recent['end'] - recent['start']).dt.days
    display_cols = ['end', 'start', 'duration_days', 'val', 'form', 'fp', 'fy', 'filed']
else:
    display_cols = ['end', 'val', 'form', 'fp', 'fy', 'filed']
recent[display_cols]

## Step 3: Cumulative differencing deep dive

Show how cumulative YTD entries are grouped by fiscal-year start date (not `fy` field)
and differenced to produce single-quarter values.

In [ ]:
# Show cumulative entries and how they are grouped
raw_with_dur = raw.dropna(subset=['start', 'end']).copy()
raw_with_dur['duration_days'] = (raw_with_dur['end'] - raw_with_dur['start']).dt.days

cumulative = raw_with_dur[
    (raw_with_dur['duration_days'] > 120) & (raw_with_dur['duration_days'] < 400)
].copy()
cumulative = cumulative.sort_values(['end', 'filed']).drop_duplicates(subset=['end'], keep='last')

# Group by start date (the fix)
print('Cumulative entries grouped by fiscal year START date:')
for start_date in sorted(cumulative['start'].dropna().unique()):
    group = cumulative[cumulative['start'] == start_date].sort_values('end')
    if group['end'].max() < pd.Timestamp('2023-01-01'):
        continue  # Skip old years for readability
    print(f'\n  FY starting {start_date.strftime("%Y-%m-%d")}:')
    for _, r in group.iterrows():
        print(f'    end={r["end"].strftime("%Y-%m-%d")} dur={r["duration_days"]}d '
              f'val=${r["val"]/1e9:.2f}B fy={r["fy"]}')

print('\n--- vs grouping by fy field (WRONG - would mix fiscal years): ---')
for fy_val in sorted(cumulative['fy'].dropna().unique()):
    if fy_val < 2023:
        continue
    group = cumulative[cumulative['fy'] == fy_val].sort_values('end')
    starts = group['start'].unique()
    print(f'\n  fy={int(fy_val)}: contains {len(starts)} different start dates: {[s.strftime("%Y-%m-%d") for s in starts]}')

## Step 4: Extracted quarterly revenue (after fix)

In [ ]:
for ticker in TEST_TICKERS:
    facts = facts_cache[ticker]
    rev = _extract_quarterly_duration(facts, 'revenue')
    rev = rev.sort_values('quarter_end')
    
    neg = (rev['value'] < 0).sum()
    recent_vals = ', '.join(
        f'${r["value"]/1e9:.1f}B' for _, r in rev.tail(4).iterrows()
    )
    print(f'{ticker}: {len(rev)} quarters, negatives={neg}')
    print(f'  Recent 4Q: {recent_vals}\n')

## Step 5: AAPL FY2025 verification

Compare pipeline output to Apple's actual reported quarterly results.

In [ ]:
# Apple's actual FY2025 quarterly revenue (from earnings releases)
expected = {
    'Q1 FY2025 (Dec 2024)': 124.30,
    'Q2 FY2025 (Mar 2025)': 95.36,
    'Q3 FY2025 (Jun 2025)': 94.04,
    'Q4 FY2025 (Sep 2025)': 102.47,
}

facts = facts_cache['AAPL']
rev = _extract_quarterly_duration(facts, 'revenue')
rev = rev.sort_values('quarter_end')
fy25 = rev[(rev['quarter_end'] >= '2024-09-28') & (rev['quarter_end'] <= '2025-12-31')]

print('AAPL FY2025 Revenue Verification:')
print(f'{"Quarter":<25} {"Pipeline ($B)":>15} {"Expected ($B)":>15} {"Match":>8}')
print('-' * 65)
for (label, exp), (_, row) in zip(expected.items(), fy25.head(4).iterrows()):
    actual = row['value'] / 1e9
    match = '✓' if abs(actual - exp) < 0.1 else '✗'
    print(f'{label:<25} {actual:>15.2f} {exp:>15.2f} {match:>8}')

## Step 6: Full pipeline output for AAPL

In [ ]:
df = _build_quarterly_dataframe(facts_cache['AAPL'])
print(f'Total rows: {len(df)}, columns: {len(df.columns)}')
print(f'Date range: {df["quarter_end_date"].min()} to {df["quarter_end_date"].max()}')
print(f'\nColumns: {list(df.columns)}')

# Show last 8 quarters
key_cols = ['quarter_end_date', 'report_date', 'revenue', 'net_income', 
            'operating_income', 'operating_cash_flow', 'total_assets']
display_df = df.sort_values('quarter_end_date').tail(8)[key_cols].copy()
for col in ['revenue', 'net_income', 'operating_income', 'operating_cash_flow', 'total_assets']:
    display_df[col] = display_df[col].apply(lambda x: f'${x/1e9:.1f}B' if pd.notna(x) else 'N/A')
display_df

## Step 7: Full pipeline test for all tickers

In [ ]:
for ticker in TEST_TICKERS:
    df = _build_quarterly_dataframe(facts_cache[ticker])
    if df is None or df.empty:
        print(f'{ticker}: No data')
        continue
    
    n_rows = len(df)
    rev_pct = df['revenue'].notna().mean() * 100
    ni_pct = df['net_income'].notna().mean() * 100
    ocf_pct = df['operating_cash_flow'].notna().mean() * 100
    ta_pct = df['total_assets'].notna().mean() * 100
    
    print(f'{ticker}: {n_rows} quarters | '
          f'rev={rev_pct:.0f}% ni={ni_pct:.0f}% ocf={ocf_pct:.0f}% ta={ta_pct:.0f}%')

## Step 8: Cash flow cumulative differencing check

Verify operating cash flow extraction handles YTD cumulative entries correctly.

In [ ]:
for ticker in ['AAPL', 'MSFT', 'AMZN']:
    facts = facts_cache[ticker]
    ocf = _extract_quarterly_duration(facts, 'operating_cash_flow')
    ocf = ocf.sort_values('quarter_end')
    
    print(f'\n{ticker} Operating Cash Flow - last 8 quarters:')
    for _, r in ocf.tail(8).iterrows():
        print(f'  {r["quarter_end"].strftime("%Y-%m-%d")}: ${r["value"]/1e9:.2f}B')